In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import io
import os
from functools import reduce

# --- UI ELEMENTS ---
upload_btn = widgets.FileUpload(accept='.csv', multiple=False, description="Upload ANA CSV")
threshold_slider = widgets.IntSlider(value=20, min=1, max=31, description='Min Days:')
process_btn = widgets.Button(description="Process Data", button_style='success')
output = widgets.Output()

def find_header_row(file_content):
    """Dynamically finds where the ANA data starts."""
    lines = file_content.decode('latin1').split('\n')
    for i, line in enumerate(lines):
        if "EstacaoCodigo" in line:
            return i
    return 14  # Fallback

def process_rainfall(b):
    with output:
        clear_output()
        if not upload_btn.value:
            print("Please upload a file first.")
            return

        # Load file from widget
        input_file = list(upload_btn.value.values())[0]
        content = input_file['content']
        filename = input_file['metadata']['name']

        skip = find_header_row(content)
        df = pd.read_csv(io.BytesIO(content), encoding='latin1', sep=';', skiprows=skip, decimal=',')

        # 1. ROBUST COLUMN SELECTION (No more [3:13] hardcoding)
        rain_cols = [f'Chuva{i:02d}' for i in range(1, 32)]
        essential_cols = ['EstacaoCodigo', 'Data', 'NivelConsistencia']

        # Keep only what exists in the file
        available_cols = [c for c in essential_cols + rain_cols if c in df.columns]
        df = df[available_cols].copy()

        # 2. DATE PROCESSING
        df['Data_dt'] = pd.to_datetime(df['Data'], dayfirst=True, errors='coerce')
        df['Ano'] = df['Data_dt'].dt.year
        df['Mes'] = df['Data_dt'].dt.month
        df = df.dropna(subset=['Ano', 'Mes'])
        df['Ano'] = df['Ano'].astype(int)
        df['Mes'] = df['Mes'].astype(int)

        # 3. CALCULATIONS
        threshold = threshold_slider.value
        df['Total Mensal'] = df[rain_cols].sum(axis=1, skipna=True)
        df['Dias com Chuva'] = df[rain_cols].apply(lambda row: (row > 0).sum(), axis=1)
        df['Registros Válidos'] = df[rain_cols].notna().sum(axis=1)

        # Logic for Raw vs Consistent
        df_raw = df[df['NivelConsistencia'] == 1].drop_duplicates(['EstacaoCodigo', 'Ano', 'Mes'])
        df_cons = df[df['NivelConsistencia'] == 2].drop_duplicates(['EstacaoCodigo', 'Ano', 'Mes'])
        df_all = df.sort_values('NivelConsistencia', ascending=False).drop_duplicates(['EstacaoCodigo', 'Ano', 'Mes'])

        # --- Helper for Stats ---
        def get_hist_monthly(source, suffix):
            valid = source[source['Registros Válidos'] >= threshold]
            if valid.empty: return pd.DataFrame()
            return valid.groupby(['EstacaoCodigo', 'Mes']).agg(**{
                f'Média Histórica{suffix}': ('Total Mensal', 'mean'),
                f'Anos Válidos{suffix}': ('Total Mensal', 'count'),
                f'Total Dias Chuva{suffix}': ('Dias com Chuva', 'sum')
            }).reset_index()

        # Generate Tables
        hm_all = get_hist_monthly(df_all, '')
        hm_raw = get_hist_monthly(df_raw, ' Bruto')
        hm_cons = get_hist_monthly(df_cons, ' Consistido')

        # Merge Results
        dfs = [d for d in [hm_all, hm_raw, hm_cons] if not d.empty]
        final_hm = reduce(lambda l, r: pd.merge(l, r, on=['EstacaoCodigo', 'Mes'], how='outer'), dfs) if dfs else pd.DataFrame()

        # 4. EXPORT (Saves to server for download)
        output_name = f"Calculos_{filename.split('.')[0]}.xlsx"
        with pd.ExcelWriter(output_name, engine='openpyxl') as writer:
            df_all.to_excel(writer, sheet_name='Dados_Processados', index=False)
            if not final_hm.empty:
                final_hm.to_excel(writer, sheet_name='Estatisticas_Mensais', index=False)

        print(f"Success! Processed {len(df_all)} records.")
        print(f"File generated: {output_name}")
        # Note: In Voilà/Binder, the file appears in the file browser or can be linked via HTML.

process_btn.on_click(process_rainfall)

# --- DISPLAY INTERFACE ---
display(widgets.HTML("<h2>Rainfall Analysis Tool (ANA)</h2>"))
display(upload_btn, threshold_slider, process_btn, output)